## Identificação das skills
Ao final, salva arquivo para análises e relatórios

In [1]:
import pandas as pd
import re
from unidecode import unidecode


### Parte 0: carrega dados

In [2]:
# Carrega os dados brutos
df = pd.read_csv('data/scp_02.csv')
print(df.shape)

(15514, 9)


In [3]:
# Carrega critérios
df_skills = pd.read_excel('data/criterios.xlsx', sheet_name='skills')
df_IA = pd.read_excel('data/criterios.xlsx', sheet_name='IA')

### Parte 1: Classificando

In [4]:
def clean_text_search(text):
    if not isinstance(text, str):
        return ''
    text = text.replace('C++', 'cplusplus').replace('c++', 'cplusplus')
    text = text.replace('C#', 'csharp').replace('c#', 'csharp')
    text = unidecode(text).lower().strip()
    text = re.sub(r'[^a-z0-9\s#+.]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text


In [5]:
df['Skills_Search'] = df['Skills'].fillna('').apply(clean_text_search)
df_skills['Sinônimos'] = df_skills['Sinônimos'].fillna('').astype(str)


In [6]:
print('Iniciando identificação de skills com sinônimos...')

for _, row in df_skills.iterrows():
    skill_name = row['Skill']
    synonyms_str = str(row['Sinônimos'])

    # Nome da coluna baseado no nome principal da skill
    col_name = 'sk_' + clean_text_search(skill_name).replace(' ', '_')
    df[col_name] = 0

    # Monta lista: skill principal + sinônimos
    terms = [skill_name]
    if synonyms_str.strip():
        terms.extend(re.split(r'[;,|]', synonyms_str))
    clean_terms = [clean_text_search(t) for t in terms if t.strip()]

    # Regra especial para R (evita match da letra isolada)
    if skill_name.strip().lower() == 'r':
        pattern = r'\b(r language|linguagem r|programacao r|r studio|rstudio)\b'
    else:
        pattern = r'\b(' + '|'.join(map(re.escape, clean_terms)) + r')\b'

    mask = df['Skills_Search'].str.contains(pattern, regex=True, na=False)
    df.loc[mask, col_name] = 1

print('Identificação concluída.')


Iniciando identificação de skills com sinônimos...


/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13074/122856229.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['Skills_Search'].str.contains(pattern, regex=True, na=False)


/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13074/122856229.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['Skills_Search'].str.contains(pattern, regex=True, na=False)


/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13074/122856229.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['Skills_Search'].str.contains(pattern, regex=True, na=False)


/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13074/122856229.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['Skills_Search'].str.contains(pattern, regex=True, na=False)


/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13074/122856229.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['Skills_Search'].str.contains(pattern, regex=True, na=False)


... (126 outputs left out)

/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13074/122856229.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = 0
/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13074/122856229.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['Skills_Search'].str.contains(pattern, regex=True, na=False)


/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13074/122856229.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = 0
/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13074/122856229.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['Skills_Search'].str.contains(pattern, regex=True, na=False)


/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13074/122856229.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = 0
/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13074/122856229.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['Skills_Search'].str.contains(pattern, regex=True, na=False)


Identificação concluída.


### Parte 2: Inteligência Artificial

In [7]:
df['IA'] = 0

for _, row in df_IA.iterrows():
    palavra_clean = clean_text_search(str(row['Palavra']))
    if not palavra_clean.strip():
        continue
    pattern = r'\b' + re.escape(palavra_clean.strip()) + r'\b'
    mask = df['Skills_Search'].str.contains(pattern, regex=True, na=False)
    df.loc[mask, 'IA'] = 1


/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13074/3989099867.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['IA'] = 0


### Parte 3: Ajusta e salva os dados

In [8]:
df.drop(columns=['Skills', 'Skills_Search', 'Local Cidade', 'Local Tipo'], inplace=True)
df.rename(columns={
    'Area': 'area',
    'Tipo': 'tipo',
    'Nivel': 'nivel',
    'Local Estado': 'estado',
    'Salario': 'salario'
}, inplace=True)


In [9]:
df.to_csv('data/scp_03.csv', index=False)